# 🏌️ Notebook 03a: Klasifikasi — Dataset Golf
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Memahami konsep klasifikasi dengan dataset yang sederhana dan intuitif
2. Melatih model Decision Tree, K-Nearest Neighbor, dan Naive Bayes
3. Memahami dan memvisualisasikan **Confusion Matrix**
4. Menghitung dan menginterpretasikan metrik evaluasi: **Accuracy, Precision, Recall, F1-Score**
5. Membuat dan menginterpretasikan **ROC Curve & AUC**
6. Membandingkan performa antar model

---
> **Dataset:** `golf.csv` — Dataset klasik machine learning yang berisi data cuaca dan keputusan bermain golf. Dataset ini ideal untuk memperkenalkan konsep klasifikasi karena fiturnya mudah dipahami secara intuitif.

| Fitur | Deskripsi |
|-------|-----------|
| `Outlook` | Kondisi langit (Sunny / Overcast / Rain) |
| `Temperature` | Suhu (Hot / Mild / Cool) |
| `Humidity` | Kelembapan (High / Normal) |
| `Windy` | Angin kencang? (True / False) |
| `Play` | **Target**: Main golf? (Yes / No) |

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, roc_auc_score
)

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11

print('Library berhasil dimuat ✅')

## 📂 2. Memuat dan Mengeksplorasi Dataset

In [ ]:
df = pd.read_csv('../../Dataset/03-Classification/golf.csv')

print(f'Ukuran dataset: {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'Kolom: {list(df.columns)}')
print()
df

In [ ]:
print('=== Tipe Data ===')
print(df.dtypes)
print()
print('=== Nilai Unik per Kolom ===')
for col in df.columns:
    print(f'  {col}: {df[col].unique()}')

## 📊 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Distribusi Target (Play)
play_counts = df['Play'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0, 0].bar(play_counts.index, play_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0, 0].set_title('Distribusi Target: Main Golf?', fontweight='bold', fontsize=13)
axes[0, 0].set_xlabel('Keputusan')
axes[0, 0].set_ylabel('Jumlah')
for i, (idx, val) in enumerate(play_counts.items()):
    axes[0, 0].text(i, val + 0.3, str(val), ha='center', fontweight='bold', fontsize=12)

# 3.2 Outlook vs Play
outlook_play = pd.crosstab(df['Outlook'], df['Play'])
outlook_play.plot(kind='bar', ax=axes[0, 1], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[0, 1].set_title('Outlook vs Keputusan Bermain', fontweight='bold', fontsize=13)
axes[0, 1].set_xlabel('Outlook')
axes[0, 1].set_ylabel('Jumlah')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].legend(title='Play')

# 3.3 Temperature vs Play
temp_play = pd.crosstab(df['Temperature'], df['Play'])
temp_play.plot(kind='bar', ax=axes[1, 0], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1, 0].set_title('Temperature vs Keputusan Bermain', fontweight='bold', fontsize=13)
axes[1, 0].set_xlabel('Temperature')
axes[1, 0].set_ylabel('Jumlah')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].legend(title='Play')

# 3.4 Humidity & Windy vs Play
hum_play = pd.crosstab(df['Humidity'], df['Play'])
hum_play.plot(kind='bar', ax=axes[1, 1], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[1, 1].set_title('Humidity vs Keputusan Bermain', fontweight='bold', fontsize=13)
axes[1, 1].set_xlabel('Humidity')
axes[1, 1].set_ylabel('Jumlah')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].legend(title='Play')

plt.suptitle('EDA — Dataset Golf', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\nRasio kelas: Yes={play_counts.get("Yes", 0)}, No={play_counts.get("No", 0)}')

## 🔧 4. Preprocessing Data

Semua fitur pada dataset golf bersifat **kategorikal** (teks). Algoritma machine learning memerlukan data numerik, sehingga kita perlu melakukan **Label Encoding** — mengkonversi setiap kategori menjadi angka.

| Fitur | Sebelum | Sesudah |
|-------|---------|---------|
| Outlook | Sunny / Overcast / Rain | 2 / 0 / 1 |
| Temperature | Hot / Mild / Cool | 1 / 2 / 0 |
| Humidity | High / Normal | 0 / 1 |
| Windy | False / True | 0 / 1 |
| Play | No / Yes | 0 / 1 |

In [ ]:
df_enc = df.copy()
le = LabelEncoder()

# Encode semua kolom kategorikal
for col in df_enc.columns:
    df_enc[col] = le.fit_transform(df_enc[col])

# Simpan mapping untuk referensi
print('=== Mapping Label Encoding ===')
for col in df.columns:
    mapping = dict(zip(df[col].unique(), le.fit(df[col]).transform(df[col].unique())))
    print(f'  {col}: {mapping}')

print()
print('Dataset setelah encoding:')
df_enc.head(10)

## ✂️ 5. Train-Test Split

In [ ]:
feature_cols = ['Outlook', 'Temperature', 'Humidity', 'Windy']
target_col = 'Play'

X = df_enc[feature_cols]
y = df_enc[target_col]

# stratify=y memastikan proporsi kelas terjaga di train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Total data   : {len(X)} baris')
print(f'Training set : {len(X_train)} baris ({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing set  : {len(X_test)} baris ({len(X_test)/len(X)*100:.0f}%)')
print()
print(f'Distribusi kelas di training set:')
print(y_train.value_counts().rename({0: 'No', 1: 'Yes'}).to_string())
print()
print(f'Distribusi kelas di testing set:')
print(y_test.value_counts().rename({0: 'No', 1: 'Yes'}).to_string())

## 🤖 6. Melatih Model Klasifikasi

Kita akan melatih **3 model** dan membandingkan performanya:
1. **Decision Tree** — Mudah diinterpretasi, bekerja seperti flowchart keputusan
2. **K-Nearest Neighbor (KNN)** — Klasifikasi berdasarkan kemiripan dengan data terdekat  
3. **Naive Bayes** — Berdasarkan probabilitas dan Teorema Bayes

In [ ]:
# --- Model 1: Decision Tree ---
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_prob = dt_model.predict_proba(X_test)[:, 1]

# --- Model 2: KNN ---
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)
knn_prob = knn_model.predict_proba(X_test)[:, 1]

# --- Model 3: Naive Bayes (Categorical) ---
nb_model = CategoricalNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_prob = nb_model.predict_proba(X_test)[:, 1]

# Ringkasan Akurasi
print('=== Akurasi Model ===')
print(f'  Decision Tree : {accuracy_score(y_test, dt_pred)*100:.1f}%')
print(f'  KNN (K=5)     : {accuracy_score(y_test, knn_pred)*100:.1f}%')
print(f'  Naive Bayes   : {accuracy_score(y_test, nb_pred)*100:.1f}%')
print()

# Visualisasi Decision Tree
plt.figure(figsize=(16, 8))
plot_tree(
    dt_model, feature_names=feature_cols, class_names=['No', 'Yes'],
    filled=True, rounded=True, fontsize=10, impurity=False
)
plt.title('Decision Tree — Dataset Golf', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Aturan pohon dalam teks
print('=== Aturan Decision Tree ===')
print(export_text(dt_model, feature_names=feature_cols))

## 🔲 7. Confusion Matrix

**Confusion Matrix** adalah tabel yang membandingkan prediksi model dengan nilai aktual.

| | Pred: No (0) | Pred: Yes (1) |
|---|---|---|
| **Aktual: No (0)** | TN — Benar diprediksi Tidak Main | FP — Salah diprediksi Main |
| **Aktual: Yes (1)** | FN — Salah diprediksi Tidak Main | TP — Benar diprediksi Main |

- **TP** = model prediksi "Main" dan memang benar Main ✅
- **TN** = model prediksi "Tidak Main" dan memang benar Tidak Main ✅  
- **FP** = model prediksi "Main" tapi aktualnya Tidak Main ❌ (False Alarm)
- **FN** = model prediksi "Tidak Main" tapi aktualnya Main ❌ (Missed)

In [ ]:
models_preds = {
    'Decision Tree': dt_pred,
    'KNN (K=5)': knn_pred,
    'Naive Bayes': nb_pred
}
labels = ['No', 'Yes']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_name, preds) in zip(axes, models_preds.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=labels, yticklabels=labels,
        ax=ax, linewidths=2, linecolor='white',
        annot_kws={'size': 16, 'fontweight': 'bold'}
    )
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{model_name}\nAccuracy: {acc*100:.1f}%', fontweight='bold', fontsize=13)
    ax.set_xlabel('Prediksi Model', fontsize=11)
    ax.set_ylabel('Nilai Aktual', fontsize=11)
    # Tambahkan label TP, TN, FP, FN
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.25, f'TN={tn}  FP={fp}  FN={fn}  TP={tp}',
            ha='center', transform=ax.transAxes, fontsize=10, color='dimgray')

plt.suptitle('Confusion Matrix — Perbandingan Model', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 📐 8. Metrik Evaluasi: Precision, Recall, F1-Score, Accuracy

### Rumus Metrik:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(dari yang diprediksi positif, berapa yang benar?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(dari yang aktual positif, berapa yang terdeteksi?)}$$

$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
for model_name, preds in models_preds.items():
    print(f'{"="*50}')
    print(f'  {model_name}')
    print(f'{"="*50}')
    print(classification_report(y_test, preds, target_names=['No', 'Yes']))
    print()

## 📈 9. ROC Curve & AUC Score

**ROC Curve** menunjukkan kemampuan model dalam membedakan antara kelas positif dan negatif pada berbagai threshold. **AUC** (Area Under the Curve) meringkas kualitas model dalam satu angka — semakin mendekati 1.0, semakin baik.

| AUC | Interpretasi |
|:---:|:---|
| 0.90 – 1.00 | Sangat Baik (Excellent) |
| 0.80 – 0.89 | Baik (Good) |
| 0.70 – 0.79 | Cukup (Fair) |
| 0.50 | Random (sama seperti tebak-tebakan) |

In [ ]:
models_probs = {
    'Decision Tree': dt_prob,
    'KNN (K=5)': knn_prob,
    'Naive Bayes': nb_prob
}
colors_roc = ['#2196F3', '#FF5722', '#4CAF50']

plt.figure(figsize=(10, 7))

for (model_name, probs), color in zip(models_probs.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2.5,
             label=f'{model_name} (AUC = {roc_auc:.3f})')

# Garis diagonal (random classifier)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.500)', alpha=0.6)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=13)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=13)
plt.title('ROC Curve — Perbandingan Model Klasifikasi Golf', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🏆 10. Perbandingan & Kesimpulan Model

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = []
for model_name, (preds, probs) in zip(
    ['Decision Tree', 'KNN (K=5)', 'Naive Bayes'],
    [(dt_pred, dt_prob), (knn_pred, knn_prob), (nb_pred, nb_prob)]
):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    results.append({
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_test, preds), 3),
        'Precision': round(precision_score(y_test, preds, zero_division=0), 3),
        'Recall': round(recall_score(y_test, preds, zero_division=0), 3),
        'F1-Score': round(f1_score(y_test, preds, zero_division=0), 3),
        'AUC': round(roc_auc, 3)
    })

df_results = pd.DataFrame(results).set_index('Model')
print('=== Ringkasan Performa Model ===')
print(df_results.to_string())

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 5))
df_results.plot(kind='bar', ax=ax, edgecolor='white', colormap='Set2', width=0.7)
ax.set_title('Perbandingan Metrik Evaluasi — Dataset Golf', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Nilai Metrik (0–1)', fontsize=12)
ax.set_ylim(0, 1.15)
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper right', fontsize=10)
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Threshold 0.8')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

print('\n✅ Model terbaik berdasarkan F1-Score:',
      df_results['F1-Score'].idxmax(),
      f"(F1 = {df_results['F1-Score'].max():.3f})")

## 📋 Rangkuman

Dataset Golf adalah dataset klasik yang digunakan untuk memperkenalkan konsep **klasifikasi** karena:
- Fiturnya sederhana dan mudah dipahami secara intuitif (cuaca)
- Target biner (Ya/Tidak) memudahkan pemahaman Confusion Matrix
- Ukuran kecil (50 baris) memudahkan visualisasi Decision Tree

### Poin Pelajaran Utama:
1. **Label Encoding** diperlukan untuk mengkonversi data kategorikal menjadi numerik
2. **Confusion Matrix** = fondasi dari semua metrik evaluasi (TP, TN, FP, FN)
3. **Accuracy** cocok jika data seimbang; gunakan **F1-Score** jika tidak seimbang
4. **ROC/AUC** memungkinkan perbandingan model secara adil tanpa memilih threshold

### Relevansi untuk Sektor Publik:
> Prinsip yang sama diterapkan saat kita membangun model untuk memprediksi **risiko satker**, **opini BPK**, atau **anomali keuangan** — bedanya hanya pada fitur dan konteksnya.

---
*Notebook: Analitika Data Keuangan Sektor Publik*